<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/doubleword.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Doubleword

[Doubleword](https://doubleword.ai) is an AI model gateway providing unified routing, management, and security for inference across multiple model providers. It exposes an OpenAI-compatible API at `https://api.doubleword.ai/v1`.

The `llamaindex-doubleword` package provides two LLM classes:

- **`DoublewordLLM`** — real-time chat and completion requests
- **`DoublewordLLMBatch`** — transparent batch API submissions via [autobatcher](https://github.com/doublewordai/llamaindex-doubleword) for reduced cost

Both support streaming, tool calling, and all models routed through Doubleword.

For more information, see the [Doubleword docs](https://docs.doubleword.ai) and the [package repository](https://github.com/doublewordai/llamaindex-doubleword).

## Installation

In [ ]:
%pip install llamaindex-doubleword llama-index

## Authentication

You can authenticate in three ways (checked in this order):

1. Pass `api_key` directly to the constructor
2. Set the `DOUBLEWORD_API_KEY` environment variable
3. Store your key in `~/.dw/credentials.toml`

In [ ]:
import os

# Option 1: environment variable
os.environ["DOUBLEWORD_API_KEY"] = "your-api-key"

## Basic Usage with DoublewordLLM

In [ ]:
from llamaindex_doubleword import DoublewordLLM

llm = DoublewordLLM(model="gpt-4o")

# Or pass the key directly:
# llm = DoublewordLLM(model="gpt-4o", api_key="your-api-key")

### Call `complete`

In [ ]:
resp = llm.complete("What is an AI gateway?")
print(resp)

### Call `chat`

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(role="user", content="Explain AI model routing in one paragraph."),
]
resp = llm.chat(messages)
print(resp)

### Streaming

In [ ]:
resp = llm.stream_chat(
    [ChatMessage(role="user", content="Tell me a short story about a router.")]
)
for r in resp:
    print(r.delta, end="")

### Tool Calling

`DoublewordLLM` supports tool calling through LlamaIndex's `AgentWorkflow`.

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow


def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b


agent = AgentWorkflow.from_tools_or_functions(
    [multiply],
    llm=DoublewordLLM(model="gpt-4o"),
)

resp = await agent.run("What is 7 times 6?")
print(resp)

## Batch Usage with DoublewordLLMBatch

`DoublewordLLMBatch` transparently submits requests through the batch API using [autobatcher](https://github.com/doublewordai/llamaindex-doubleword), which can reduce costs significantly. It has the same interface as `DoublewordLLM`.

In [ ]:
from llamaindex_doubleword import DoublewordLLMBatch

llm_batch = DoublewordLLMBatch(model="gpt-4o")

resp = llm_batch.complete("Summarize the benefits of batch processing.")
print(resp)

## Using with LlamaIndex Settings

You can set Doubleword as the default LLM globally:

In [ ]:
from llama_index.core import Settings

Settings.llm = DoublewordLLM(model="gpt-4o")